In [14]:
import os
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

In [18]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")


if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")


OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-


In [19]:
openai= OpenAI()

anthropic = OpenAI(api_key=anthropic_api_key, base_url="https://api.anthropic.com/v1")

In [22]:
def call_gpt(input_prompt):
    messages = [{"role": "system", "content": "You are a helpful assistant."},{"role": "user", "content": input_prompt}]
    response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
    return response.choices[0].message.content

In [46]:
def stream_gpt(input_prompt):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": input_prompt}
    ]

    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        stream=True
    )

    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or "" #If content is empty or None, use an empty string instead.
        yield result

In [24]:
def call_Claude(input_prompt):
    messages = [{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": input_prompt}]
    response = anthropic.chat.completions.create(model="claude-sonnet-4-5-20250929", messages=messages)
    return response.choices[0].message.content

In [57]:
def stream_Claude(input_prompt):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."}, 
        {"role": "user", "content": input_prompt}
        ]
    stream = anthropic.chat.completions.create(
        model="claude-sonnet-4-5-20250929",
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or "" #If content is empty or None, use an empty string instead.
        yield result

In [58]:
def stream_model(prompt, model):
    if model=="GPT":
        result = stream_gpt(prompt)
    elif model=="anthropic":
        result = stream_Claude(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [ ]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(["GPT", "Claude"], label="Select model", value="GPT")
message_output = gr.Markdown(label="Response:")

message_input = gr.Textbox(label="Your message:", info="Enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(["GPT", "Claude"], label="Select model", value="GPT")
message_output = gr.Markdown(label="Response:")

gr.Interface(
    fn=stream_model,
    inputs=[message_input, model_selector],
    outputs=message_output,
    examples=[
        ["Explain the Transformer architecture to a layperson", "GPT"],
        ["Explain the Transformer architecture to an aspiring AI engineer", "Claude"]
    ],
    title="LLM TEST",
    flagging_mode="never",
).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [61]:
gr.close_all()

Closing server running on port: 7861
Closing server running on port: 7860
